### Groenbeeld repository clonen

In [1]:
!pip install owslib > /dev/null 2>&1

In [2]:
import os
import re
import math
import time
import asyncio
import gc
from io import BytesIO
import urllib.parse
import geopandas as gpd
import aiohttp
from PIL import Image
import argparse
from pathlib import Path
import numpy as np
import rasterio as rio
from rasterio.mask import mask
from rasterio.merge import merge
import matplotlib.pyplot as plt
import pandas as pd
from skimage.segmentation import slic
from scipy.ndimage import mean
from sklearn.metrics import silhouette_score
from sklearn.cluster import MiniBatchKMeans, Birch
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, normalize
import xml.etree.ElementTree as ET
from matplotlib.colors import ListedColormap

In [3]:
if os.path.exists('Groenbeeld'):
  !rm -rf 'Groenbeeld'
  print("Groenbeeld directory removed.")
else:
  print("Groenbeeld directory does not exist. Skipping removal.")

Groenbeeld directory does not exist. Skipping removal.


In [4]:
!git clone https://github.com/bart314/Groenbeeld.git
os.chdir('Groenbeeld')
!git checkout clustering
!ls

Cloning into 'Groenbeeld'...
remote: Enumerating objects: 231, done.
remote: Counting objects: 100% (92/92), done.
remote: Compressing objects: 100% (76/76), done.
remote: Total 231 (delta 47), reused 38 (delta 15), pack-reused 139 (from 1)
Receiving objects: 100% (231/231), 13.37 MiB | 23.94 MiB/s, done.
Resolving deltas: 100% (111/111), done.
Branch 'clustering' set up to track remote branch 'clustering' from 'origin'.
Switched to a new branch 'clustering'
Data	    NDVI-SWF.yaml  README.md  targets.txt
LICENSE.md  outdir	   Scripts    Tutorial_grondwaterstand


In [5]:
## updaten van de targets.txt (Alle buurten van Bolsward gepakt!)
%%writefile /content/Groenbeeld/targets.txt
BU19000000
BU19000001
BU19000002
BU19000003
BU19000004

Overwriting /content/Groenbeeld/targets.txt


In [6]:
!python3 'Scripts/Municipality_Surveyor_V4.py' --limit 5

Initializing CIR WMTS service (2022)...
Initializing RGB WMTS service (2022)...
Initializing BRT WMTS service...
Initializing BRT-A Background service (Topotijdreis 2022)...
Initializing BGT WMTS service...
WMTS initialized. Resolution: 0.21m/px
Initializing KEA WMS service...
Connecting to Sogelink Server for 7 KEA layers...
 SUCCESS: Connected. Layer 'gevoelstemperatuur_2022' found.
 SUCCESS: Connected. Layer 'RMK_BKB_AHN4' found.
 SUCCESS: Connected. Layer 'RMK_SchaduwGrijs_AHN4' found.
 SUCCESS: Connected. Layer 'RMK_SchaduwGroen_AHN4' found.
 SUCCESS: Connected. Layer 'BGV_ONO_2024' found.
 SUCCESS: Connected. Layer 'BGV_landbedekking_2024' found.
 SUCCESS: Connected. Layer 'RMK_30percBKB_AHN4' found.
Fetching specific neighborhoods (2022 boundaries): BU19000000, BU19000001, BU19000002, BU19000003, BU19000004...
Found 5 neighborhood geometries.
[1/2] Processing Bolsward binnen De Wallen (BU19000000)...
  Generating categorical BGT raster (Peildatum: 2022-05-15T12:00:00Z)...
  Gene

### Clipping on CBS neighbourhoods

In [7]:
def fetch_buurten(MUNICIPALITY_NAME='Sudwest-Fryslan', MUNICIPALITY_CODE='GM1900'):
        """Fetch all buurten for the municipality using a server-side filter."""
        print(f"Fetching all buurten for {MUNICIPALITY_NAME} ({MUNICIPALITY_CODE})...")
        import requests

        # OGC Filter for GM1900 and land-only (water=NEE)
        ogc_filter = f"""
        <ogc:Filter xmlns:ogc='http://www.opengis.net/ogc'>
            <ogc:And>
                <ogc:PropertyIsEqualTo>
                    <ogc:PropertyName>gemeentecode</ogc:PropertyName>
                    <ogc:Literal>{MUNICIPALITY_CODE}</ogc:Literal>
                </ogc:PropertyIsEqualTo>
                <ogc:PropertyIsEqualTo>
                    <ogc:PropertyName>water</ogc:PropertyName>
                    <ogc:Literal>NEE</ogc:Literal>
                </ogc:PropertyIsEqualTo>
            </ogc:And>
        </ogc:Filter>
        """
        WFS_URL = "https://service.pdok.nl/cbs/wijkenbuurten/2023/wfs/v1_0"
        encoded_filter = urllib.parse.quote(ogc_filter.strip())
        url = f"{WFS_URL}?request=GetFeature&service=WFS&version=2.0.0&typeName=wijkenbuurten:buurten&outputFormat=application/json&filter={encoded_filter}"

        response = requests.get(url, timeout=30)
        if response.status_code != 200:
            raise Exception(f"Failed to fetch buurten: {response.status_code}")

        df = gpd.read_file(BytesIO(response.content))

        if df.empty:
            print(f"Error: No land-based neighborhoods found for {MUNICIPALITY_CODE}.")
            return df

        print(f"Found {len(df)} land-based buurten for {MUNICIPALITY_NAME}.")
        return df


def clip_feature_tif(indir, outdir, cbs_buurten,
                     file_types=["NDVI", "CHM", "CIR", "BGT_CAT", "KEA_CAT_gevoelstemperatuur_2022", "KEA_CAT_BGV_landbedekking_2024"],
                     target_file='/content/Groenbeeld/targets.txt'):
    outdir_path = Path(outdir)
    outdir_path.mkdir(parents=True, exist_ok=True)

    indir_path = Path(indir)
    with open(target_file, 'r') as f:
        targets = [line.strip() for line in f if line.strip()]
        for t in targets:
            print()
            print(f'Clipping feature .tif files on {t} buurt...')
            cbs_buurt = cbs_buurten[cbs_buurten['buurtcode'] == t]
            if cbs_buurt.empty:
                print(f"Warning: CBS buurt with code {t} not found. Skipping.")
                continue
            for file_type in file_types:
                file_glob_pattern = f"*{t}*_{file_type}.tif"
                found_files = list(indir_path.glob(file_glob_pattern))

                if not found_files:
                    print(f"Warning: {file_type} file not found for target {t} with pattern '{file_glob_pattern}'. Skipping.")
                    continue
                if len(found_files) > 1:
                    print(f"Warning: Multiple {file_type} files found for target {t} with pattern '{file_glob_pattern}'. Using the first one: {found_files[0].name}.")

                file_path = found_files[0]
                print(f"Processing {file_path.name} for target {t}...")

                with rio.open(file_path) as src:
                    buurt_reprojected = cbs_buurt.to_crs(src.crs)
                    geometrie = [buurt_reprojected.geometry.iloc[0]]

                    out_image, out_transform = mask(src, geometrie, crop=True)
                    out_meta = src.meta.copy()

                    out_meta.update({
                        "height": out_image.shape[1],
                        "width": out_image.shape[2],
                        "transform": out_transform
                    })

                original_filename = file_path.name
                base_name, ext = original_filename.rsplit('.', 1)
                new_filename = f"{base_name}_clipped.{ext}"
                output_path = outdir_path / new_filename

                with rio.open(output_path, "w", **out_meta) as dest:
                    dest.write(out_image)
                print(f"Saved {t} clipped file to {output_path}")

In [8]:
cbs_buurten = fetch_buurten()

Fetching all buurten for Sudwest-Fryslan (GM1900)...
Found 155 land-based buurten for Sudwest-Fryslan.


In [9]:
## Clippen van de rechthoeken op de exacte buurt geometrieen,
## om deze later tot een .tif bestand te maken na prediction
indir = "outdir/municipality_survey_v4"
outdir = "outdir/municipality_survey_v4/preprocessed/"

clip_feature_tif(indir, outdir, cbs_buurten)


Clipping feature .tif files on BU19000000 buurt...
Processing BU19000000_Bolsward_binnen_De_Wallen_NDVI.tif for target BU19000000...
Saved BU19000000 clipped file to outdir/municipality_survey_v4/preprocessed/BU19000000_Bolsward_binnen_De_Wallen_NDVI_clipped.tif
Processing BU19000000_Bolsward_binnen_De_Wallen_CHM.tif for target BU19000000...
Saved BU19000000 clipped file to outdir/municipality_survey_v4/preprocessed/BU19000000_Bolsward_binnen_De_Wallen_CHM_clipped.tif
Processing BU19000000_Bolsward_binnen_De_Wallen_CIR.tif for target BU19000000...
Saved BU19000000 clipped file to outdir/municipality_survey_v4/preprocessed/BU19000000_Bolsward_binnen_De_Wallen_CIR_clipped.tif
Processing BU19000000_Bolsward_binnen_De_Wallen_BGT_CAT.tif for target BU19000000...
Saved BU19000000 clipped file to outdir/municipality_survey_v4/preprocessed/BU19000000_Bolsward_binnen_De_Wallen_BGT_CAT_clipped.tif
Processing BU19000000_Bolsward_binnen_De_Wallen_KEA_CAT_gevoelstemperatuur_2022.tif for target BU1

### Run models on multiple neighbourhoods and combine into one .tif file

In [10]:
"""
Dit gedeelte doet de volgende stappen:
  1. Voor een specifieke buurt een model trainen (GMM, Kmeans, Birch) en dit getrainde model returnen
  2. Alle buurten met dit getrainde model predicten en wegschrijven als .tif inclusief kleuring van de clusters
  3. Alle losse buurten samenvoegen tot één .tif bestand.
"""

'\nDit gedeelte doet de volgende stappen:\n  1. Voor een specifieke buurt een model trainen (GMM, Kmeans, Birch) en dit getrainde model returnen\n  2. Alle buurten met dit getrainde model predicten en wegschrijven als .tif inclusief kleuring van de clusters\n  3. Alle losse buurten samenvoegen tot één .tif bestand. \n'

#### Helper functions

In [11]:
def find_knee(x, y):
    """Simple heuristic to find the 'knee' in an elbow curve."""
    v = np.array([x[-1] - x[0], y[-1] - y[0]])
    v_norm = v / np.linalg.norm(v)

    distances = []
    for i in range(len(x)):
        p = np.array([x[i] - x[0], y[i] - y[0]])
        dist = np.linalg.norm(p - np.dot(p, v_norm) * v_norm)
        distances.append(dist)
    return x[np.argmax(distances)]


def process_neighborhood(indir, target_name):
    indir_path = Path(indir)
    ndvi_path = next(indir_path.glob(f"*{target_name}*_NDVI_clipped.tif"), None)
    chm_path = next(indir_path.glob(f"*{target_name}*_CHM_clipped.tif"), None)
    cir_path = next(indir_path.glob(f"*{target_name}*_CIR_clipped.tif"), None)

    print(f"Loading data for {target_name}...")

    # -----------------------------
    # NDVI
    # -----------------------------
    with rio.open(ndvi_path) as src:
        ndvi = src.read(1)
        profile = src.profile
        height, width = ndvi.shape
        ndvi_mask = (ndvi == 255)

    # -----------------------------
    # CHM
    # -----------------------------
    with rio.open(chm_path) as src:
        chm = src.read(1, out_shape=(height, width))
        chm_mask = (chm < -9000)

    # -----------------------------
    # CIR (NIR + RED)
    # -----------------------------
    with rio.open(cir_path) as src:
        nir = src.read(1, out_shape=(height, width))
        red = src.read(2, out_shape=(height, width))
        cir_nodata = src.nodata or 255
        cir_mask = (nir == cir_nodata) | (red == cir_nodata)

    # -----------------------------
    # Combined mask
    # -----------------------------
    invalid_mask = ndvi_mask | chm_mask | cir_mask
    valid_indices = np.where(~invalid_mask)

    print(f"Total valid pixels: {len(valid_indices[0])}")

    # -----------------------------
    # Feature matrix
    # -----------------------------
    features = np.column_stack([
        ndvi[valid_indices],
        chm[valid_indices],
        nir[valid_indices],
        red[valid_indices]
    ]).astype(np.float32)

    return features, valid_indices, invalid_mask, (height, width), profile


def process_neighborhood_v2(indir, target_name):
    ## version wuth more features
    indir_path = Path(indir)
    ndvi_path = next(indir_path.glob(f"*{target_name}*_NDVI_clipped.tif"), None)
    chm_path = next(indir_path.glob(f"*{target_name}*_CHM_clipped.tif"), None)
    cir_path = next(indir_path.glob(f"*{target_name}*_CIR_clipped.tif"), None)

    ## extra features
    bgt_cat_path = next(indir_path.glob(f"*{target_name}*_BGT_CAT_clipped.tif"), None)
    gevoelstemp_2022 = next(indir_path.glob(f"*{target_name}*_KEA_CAT_gevoelstemperatuur_2022_clipped.tif"), None)
    landbedekking_2024 = next(indir_path.glob(f"*{target_name}*_KEA_CAT_BGV_landbedekking_2024_clipped.tif"), None)

    print(f"Loading data for {target_name}...")

    # -----------------------------
    # NDVI
    # -----------------------------
    with rio.open(ndvi_path) as src:
        ndvi = src.read(1)
        profile = src.profile
        height, width = ndvi.shape
        ndvi_mask = (ndvi == 255)

    # -----------------------------
    # CHM
    # -----------------------------
    with rio.open(chm_path) as src:
        chm = src.read(1, out_shape=(height, width))
        chm_mask = (chm < -9000)

    # -----------------------------
    # CIR (NIR + RED)
    # -----------------------------
    with rio.open(cir_path) as src:
        nir = src.read(1, out_shape=(height, width))
        red = src.read(2, out_shape=(height, width))
        cir_nodata = src.nodata or 255
        cir_mask = (nir == cir_nodata) | (red == cir_nodata)

    # -----------------------------
    # BGT CAT
    # -----------------------------
    with rio.open(bgt_cat_path) as src:
        bgt_cat = src.read(1)
        profile = src.profile
        height, width = bgt_cat.shape
        bgt_cat_mask = (bgt_cat == src.nodata)

    # -----------------------------
    # KEA CAT gevoelstemp 2022
    # -----------------------------
    with rio.open(gevoelstemp_2022) as src:
        gevoelstemp = src.read(1)
        profile = src.profile
        height, width = gevoelstemp.shape
        gevoelstemp_mask = (gevoelstemp == src.nodata)

    # -----------------------------
    # KEA CAT BGV landbedekking 2024
    # -----------------------------
    with rio.open(landbedekking_2024) as src:
        landbedekking = src.read(1)
        profile = src.profile
        height, width = landbedekking.shape
        landbedekking_mask = (landbedekking == src.nodata)

    # -----------------------------
    # Combined mask
    # -----------------------------
    invalid_mask = ndvi_mask | chm_mask | cir_mask | bgt_cat_mask | gevoelstemp_mask | landbedekking_mask
    valid_indices = np.where(~invalid_mask)

    print(f"Total valid pixels: {len(valid_indices[0])}")

    # -----------------------------
    # Feature matrix
    # -----------------------------
    features = np.column_stack([
        ndvi[valid_indices],
        chm[valid_indices],
        nir[valid_indices],
        red[valid_indices],
        bgt_cat[valid_indices],
        gevoelstemp[valid_indices],
        landbedekking[valid_indices]
    ]).astype(np.float32)

    return features, valid_indices, invalid_mask, (height, width), profile


def combine_cluster_tifs(input_dir, model_type="GMM", output_filename="combined_clusters.tif"):
    input_path = Path(input_dir)
    tif_files = list(input_path.glob(f"*_clusters_{model_type}.tif"))

    if not tif_files:
        print(f"No {model_type} cluster .tif files found in {input_path}")
        return

    print(f"Found {len(tif_files)} .tif files to combine:")
    for f in tif_files:
        print(f"  - {f.name}")

    src_files_to_mosaic = []
    for fp in tif_files:
        src_files_to_mosaic.append(rio.open(fp))

    first_src_colormap = src_files_to_mosaic[0].colormap(1) if src_files_to_mosaic[0].colormap(1) else None

    print("Merging files...")
    mosaic, out_transform = merge(src_files_to_mosaic)

    out_meta = src_files_to_mosaic[0].meta.copy()
    out_meta.update(
        {
            "driver": "GTiff",
            "height": mosaic.shape[1],
            "width": mosaic.shape[2],
            "transform": out_transform,
            "crs": src_files_to_mosaic[0].crs,
            "nodata": 255,
        }
    )

    for src in src_files_to_mosaic:
        src.close()

    output_path = input_path / f"{model_type}_{output_filename}"
    print(f"Writing combined .tif file to {output_path}")
    with rio.open(output_path, "w", **out_meta) as dest:
        dest.write(mosaic)
        if first_src_colormap:
            dest.write_colormap(1, first_src_colormap)

    print(f"Combined .tif file saved successfully.")

#### GMM model

In [17]:
def Train_GMM(features, subset_size=1000000, covariance_type='full', random_state=42):
    print("="*40)
    print("GMM: train on subset sample buurt → return trained model")
    print("="*40)

    # -----------------------------
    # 1. Sample
    # -----------------------------
    idx = np.random.choice(features.shape[0], size=subset_size, replace=False)
    X_train = features[idx]
    train_perc = round(X_train.shape[0]/features.shape[0]*100, 2)

    # -----------------------------
    # 2. Scaling
    # -----------------------------
    scaler = StandardScaler()
    feature_scaled = scaler.fit_transform(X_train)

    # -----------------------------
    # 3. Train (find optimal K)
    # -----------------------------
    print(f"Training on sample size: {X_train.shape[0]}/{features.shape[0]} ({train_perc}%)")

    bics_results = []
    ks = range(2, 10)

    print(f"Finding optimal K-value ({min(ks)}-{max(ks)})...")
    for k in ks:
        gmm = GaussianMixture(
            n_components=k,
            covariance_type=covariance_type,
            random_state=random_state
        )
        gmm.fit(feature_scaled)
        bic = gmm.bic(feature_scaled)
        bics_results.append(bic)
        print(f"  K={k}, Bic={bic:.2f}")

    optimal_k = find_knee(list(ks), bics_results)
    print(f'optimal K-value: {optimal_k}')

    # -----------------------------
    # 4. Train (final model)
    # -----------------------------
    gmm_final = GaussianMixture(
        n_components=optimal_k,
        covariance_type=covariance_type,
        random_state=random_state
    )
    gmm_final.fit(feature_scaled)
    print(f'Final model trained, which can be used to cluster further neighbourhoods')
    return gmm_final, scaler


def Predict_GMM(gmm_model, scaler, target_file, input_dir, output_dir, chunk_size=5000000):
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    with open(target_file, 'r') as f:
        targets = [line.strip() for line in f if line.strip()]

    for t in targets:
        features, valid_indices, invalid_mask, raster_shape, profile = process_neighborhood_v2(input_dir, t)
        print("="*40)
        print(f'Predicting on {t}...')

        # -----------------------------
        # 1. Predict pixels (chunked)
        # -----------------------------
        n_samples = features.shape[0]
        labels = np.empty(n_samples, dtype=np.int32)

        print(f"Predicting {n_samples} pixels in chunks...")

        for i in range(0, n_samples, chunk_size):
            end = min(i + chunk_size, n_samples)

            X_chunk = features[i:end]
            X_chunk_scaled = scaler.transform(X_chunk)
            labels[i:end] = gmm_model.predict(X_chunk_scaled)
            print(f"  Processed {end}/{n_samples}")

        # -----------------------------
        # 2. Reconstruct raster
        # -----------------------------
        label_raster = np.full(raster_shape, -1, dtype=np.int32)
        label_raster[valid_indices] = labels

        # -----------------------------
        # 5. Export
        # -----------------------------
        out_profile = profile.copy()
        out_profile.update(
            dtype=rio.uint8,
            count=1,
            nodata=255,
            compress='lzw'
        )

        output_path = Path(output_dir)/f"{t}_clusters_GMM.tif"

        palette = [(31,119,180),(255,127,14),(44,160,44),(214,39,40),(148,103,189),
                   (140,86,75),(227,119,194),(127,127,127),(188,189,34),(23,190,207)]

        colormap = {
            int(i): palette[i % len(palette)] + (255,)
            for i in np.unique(labels)
        }

        with rio.open(output_path, "w", **out_profile) as dst:
            dst.write(label_raster.astype(np.uint8), 1)
            dst.write_colormap(1, colormap)

        print(f"Saved → {output_path}")
        print("="*40)
        print('\n')

#### KMEANS model

In [14]:
def Train_KMEANS(features, subset_size=2500000, random_state=42):
    print("="*40)
    print("Kmeans: train on subset sample buurt → return trained model")
    print("="*40)

    # -----------------------------
    # 1. Sample
    # -----------------------------
    idx = np.random.choice(features.shape[0], size=subset_size, replace=False)
    X_train = features[idx]

    # -----------------------------
    # 2. Scaling
    # -----------------------------
    scaler = StandardScaler()
    feature_scaled = scaler.fit_transform(X_train)

    # -----------------------------
    # 3. Train (find optimal K)
    # -----------------------------
    print(f"Training Kmeans: ")

    inertias_results = []
    ks = range(2, 10)

    print(f"  Finding optimal K-value ({min(ks)}-{max(ks)})...")
    for k in ks:
        kmeans = MiniBatchKMeans(
            n_clusters=k,
            batch_size=1024,
            random_state=random_state
        )
        kmeans.fit(feature_scaled)
        inertias_results.append(kmeans.inertia_)
        print(f"  K={k}, Inertia={kmeans.inertia_:.2f}")

    optimal_k = find_knee(list(ks), inertias_results)
    print(f'  Optimal K-value: {optimal_k}')

    # -----------------------------
    # 4. Train (final model)
    # -----------------------------
    kmeans_final = MiniBatchKMeans(
        n_clusters=optimal_k,
        batch_size=1024,
        random_state=random_state
    )
    kmeans_final.fit(feature_scaled)
    print(f'Final model trained, which can be used to cluster further neighbourhoods')
    return kmeans_final, scaler


def Predict_KMEANS(kmeans_model, scaler, target_file, input_dir, output_dir, chunk_size=5000000):
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    with open(target_file, 'r') as f:
        targets = [line.strip() for line in f if line.strip()]

    for t in targets:
        features, valid_indices, invalid_mask, raster_shape, profile = process_neighborhood_v2(input_dir, t)
        print("="*40)
        print(f'Predicting on {t}...')

        # -----------------------------
        # 1. Predict pixels (chunked)
        # -----------------------------
        n_samples = features.shape[0]
        labels = np.empty(n_samples, dtype=np.int32)

        print(f"Predicting {n_samples} pixels in chunks...")

        for i in range(0, n_samples, chunk_size):
            end = min(i + chunk_size, n_samples)

            X_chunk = features[i:end]
            X_chunk_scaled = scaler.transform(X_chunk)
            labels[i:end] = kmeans_model.predict(X_chunk_scaled)
            print(f"  Processed {end}/{n_samples}")

        # -----------------------------
        # 2. Reconstruct raster
        # -----------------------------
        label_raster = np.full(raster_shape, -1, dtype=np.int32)
        label_raster[valid_indices] = labels

        # -----------------------------
        # 5. Export
        # -----------------------------
        out_profile = profile.copy()
        out_profile.update(
            dtype=rio.uint8,
            count=1,
            nodata=255,
            compress='lzw'
        )

        output_path = Path(output_dir)/f"{t}_clusters_KMEANS.tif"

        palette = [(31,119,180),(255,127,14),(44,160,44),(214,39,40),(148,103,189),
                   (140,86,75),(227,119,194),(127,127,127),(188,189,34),(23,190,207)]

        colormap = {
            int(i): palette[i % len(palette)] + (255,)
            for i in np.unique(labels)
        }

        with rio.open(output_path, "w", **out_profile) as dst:
            dst.write(label_raster.astype(np.uint8), 1)
            dst.write_colormap(1, colormap)

        print(f"Saved → {output_path}")
        print("="*40)
        print('\n')

#### Birch model

In [15]:
def Train_BIRCH(features, subset_size=200000, random_state=42):
    print("="*40)
    print("Birch: train on subset sample buurt → return trained model")
    print("="*40)

    # -----------------------------
    # 1. Sample
    # -----------------------------
    idx = np.random.choice(features.shape[0], size=subset_size, replace=False)
    X_train = features[idx]

    # -----------------------------
    # 2. Scaling
    # -----------------------------
    scaler = StandardScaler()
    feature_scaled = scaler.fit_transform(X_train)

    # -----------------------------
    # 3. Train (find optimal branching_factor and threshold)
    # -----------------------------
    print("Finding optimal branching_factor and threshold using Silhouette Score...")

    best_score = -1
    best_birch_model = None
    best_params = {}

    # Define ranges for parameters to search
    branching_factors = [100]
    thresholds = [2, 2.05, 2.1, 2.15, 2.2, 2.25, 2.3, 2.35, 2.4, 2.45, 2.5]

    for t in thresholds:
        for bf in branching_factors:
            try:
                birch = Birch(
                    threshold=t,
                    branching_factor=bf,
                    n_clusters=None,
                    )
                birch.fit(feature_scaled)
                labels = birch.predict(feature_scaled)
                unique_labels = np.unique(labels)
                print(f"  Branching_factor={bf}, threshold={t}, Silhouette score: {score:.4f}...")
                if score > best_score:
                    best_score = score
                    best_birch_model = birch
                    best_params = {'branching_factor': bf, 'threshold': t}

            except Exception as e:
                print(f"Error during Birch training for bf={bf}, t={t}: {e}")

    num_clusters = len(np.unique(best_birch_model.predict(feature_scaled)))
    print(f'Optimal Birch parameters found: {best_params} with Silhouette Score: {best_score:.4f} and {num_clusters} unique clusters')
    print(f'Final model trained, which can be used to cluster further neighbourhoods')
    return best_birch_model, scaler


def Predict_BIRCH(birch_model, scaler, target_file, input_dir, output_dir, chunk_size=5000000):
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    with open(target_file, 'r') as f:
        targets = [line.strip() for line in f if line.strip()]

    for t in targets:
        features, valid_indices, invalid_mask, raster_shape, profile = process_neighborhood_v2(input_dir, t)
        print("="*40)
        print(f'Predicting on {t}...')

        # -----------------------------
        # 1. Predict pixels (chunked)
        # -----------------------------
        n_samples = features.shape[0]
        labels = np.empty(n_samples, dtype=np.int32)

        print(f"Predicting {n_samples} pixels in chunks...")

        for i in range(0, n_samples, chunk_size):
            end = min(i + chunk_size, n_samples)

            X_chunk = features[i:end]
            X_chunk_scaled = scaler.transform(X_chunk)
            labels[i:end] = birch_model.predict(X_chunk_scaled)
            print(f"  Processed {end}/{n_samples}")

        # -----------------------------
        # 2. Reconstruct raster
        # -----------------------------
        label_raster = np.full(raster_shape, -1, dtype=np.int32)
        label_raster[valid_indices] = labels

        # -----------------------------
        # 5. Export
        # -----------------------------
        out_profile = profile.copy()
        out_profile.update(
            dtype=rio.uint8,
            count=1,
            nodata=255,
            compress='lzw'
        )

        output_path = Path(output_dir)/f"{t}_clusters_BIRCH.tif"

        palette = [(31,119,180),(255,127,14),(44,160,44),(214,39,40),(148,103,189),
                   (140,86,75),(227,119,194),(127,127,127),(188,189,34),(23,190,207)]

        colormap = {
            int(i): palette[i % len(palette)] + (255,)
            for i in np.unique(labels) if i != -1 # Exclude nodata (-1) from colormap assignment
        }

        with rio.open(output_path, "w", **out_profile) as dst:
            dst.write(label_raster.astype(np.uint8), 1)
            dst.write_colormap(1, colormap)

        print(f"Saved → {output_path}")
        print("="*40)
        print('\n')

#### Train, predict and combine output .TIF

In [18]:
## Training GMM
indir = "outdir/municipality_survey_v4/preprocessed"
features_train_gmm, _, _, (_, _), _ = process_neighborhood_v2(indir, "BU19000001")
gmm_model, gmm_scalar = Train_GMM(features_train_gmm)

Loading data for BU19000001...
Total valid pixels: 12006055
GMM: train on subset sample buurt → return trained model
Training on sample size: 1000000/12006055 (8.33%)
Finding optimal K-value (2-9)...
  K=2, Bic=12831927.70
  K=3, Bic=-764447.27
  K=4, Bic=1289086.78
  K=5, Bic=-5721446.64
  K=6, Bic=-9512122.26
  K=7, Bic=-9703129.11
  K=8, Bic=-12668076.15
  K=9, Bic=-12948963.33
optimal K-value: 3
Final model trained, which can be used to cluster further neighbourhoods


In [19]:
## Training KMEANS
indir = "outdir/municipality_survey_v4/preprocessed"
features_train_kmeans, _, _, (_, _), _ = process_neighborhood_v2(indir, "BU19000001")
kmeans_model, kmeans_scalar = Train_KMEANS(features_train_kmeans)

Loading data for BU19000001...
Total valid pixels: 12006055
Kmeans: train on subset sample buurt → return trained model
Training Kmeans: 
  Finding optimal K-value (2-9)...
  K=2, Inertia=13390186.00
  K=3, Inertia=11202517.00
  K=4, Inertia=8269406.00
  K=5, Inertia=7700775.00
  K=6, Inertia=6638959.00
  K=7, Inertia=5892064.00
  K=8, Inertia=5505519.50
  K=9, Inertia=5294925.50
  Optimal K-value: 4
Final model trained, which can be used to cluster further neighbourhoods


In [ ]:
## Training BIRCH (Warning: takes long to run with 200K sample adjust to speed up!)
indir = "outdir/municipality_survey_v4/preprocessed"
features_train_birch, _, _, (_, _), _ = process_neighborhood_v2(indir, "BU19000001")
birch_model, birch_scalar = Train_BIRCH(features_train_birch)

In [ ]:
## Prediction GMM
outdir_pred_gmm = "/content/Groenbeeld/outdir/exploratory_analysis/GMM_results/"
indir = "outdir/municipality_survey_v4/preprocessed"
target_filepath = "targets.txt"

Predict_GMM(gmm_model, gmm_scalar, target_filepath, indir, outdir_pred_gmm)

In [ ]:
## Prediction KMEANS
outdir_pred_kmeans = "/content/Groenbeeld/outdir/exploratory_analysis/KMEANS_results/"
indir = "outdir/municipality_survey_v4/preprocessed"
target_filepath = "targets.txt"

Predict_KMEANS(kmeans_model, kmeans_scalar, target_filepath, indir, outdir_pred_kmeans)

In [ ]:
## Prediction BIRCH
outdir_pred_birch = "/content/Groenbeeld/outdir/exploratory_analysis/BIRCH_results/"
indir = "outdir/municipality_survey_v4/preprocessed"
target_filepath = "targets.txt"

Predict_BIRCH(birch_model, birch_scalar, target_filepath, indir, outdir_pred_birch)

In [ ]:
## Combine for each model the output .tif
GMM_results_path = "outdir/exploratory_analysis/GMM_results"
KMEANS_results_path = "outdir/exploratory_analysis/KMEANS_results"
#BIRCH_results_path = "outdir/exploratory_analysis/BIRCH_results"

combine_cluster_tifs(KMEANS_results_path, model_type="KMEANS")
combine_cluster_tifs(GMM_results_path, model_type="GMM")
#combine_cluster_tifs(BIRCH_results_path, model_type="BIRCH")

## Test

### Run models

In [ ]:
!python3 "/content/Groenbeeld/Scripts/Exploratory_Clustering_GMM.py" --indir "/content/Groenbeeld/outdir/municipality_survey_v4/preprocessed" --target "BU19000001"

Loaded 5 targets from targets.txt

Processing Target: BU19000001
Loading data for BU19000001...
Total valid pixels: 12006055
GMM: train on subset sample → predict on all data
Training on sample size: 1000000/12006055 (8.33%)
Finding optimal K-value (2-9)...
  K=2, Bic=11105509.46
  K=3, Bic=6631957.65
  K=4, Bic=5794924.05
  K=5, Bic=-1072701.13
  K=6, Bic=-5326187.88
  K=7, Bic=-6817429.79
  K=8, Bic=-7106740.27
  K=9, Bic=-9844566.66
optimal K-value: 6
Predicting 12006055 pixels in chunks...
  Processed 5000000/12006055
  Processed 10000000/12006055
  Processed 12006055/12006055
Saved → outdir/exploratory_analysis/GMM_results/BU19000001_clusters_GMM.tif

Processing Target: BU19000000
Loading data for BU19000000...
Total valid pixels: 3666355
GMM: train on subset sample → predict on all data
Training on sample size: 1000000/3666355 (27.28%)
Finding optimal K-value (2-9)...
  K=2, Bic=9858627.10
  K=3, Bic=9419614.99
  K=4, Bic=8306317.35
  K=5, Bic=2737405.54
  K=6, Bic=1273643.82
  K

In [ ]:
!python3 "/content/Groenbeeld/Scripts/Exploratory_Clustering_Birchmodel.py" --branch 100 --threshold 2.15

In [ ]:
!python3 "/content/Groenbeeld/Scripts/Exploratory_Clustering_V2.py"

### Feature inspection

In [ ]:
"""
Hier kun je nieuwe features toevoegen en inspecteren,
voor nu heb ik drie extra features toegevoegd.
"""

In [ ]:
def load_features(indir="/content/Groenbeeld/outdir/municipality_survey_v4", target_name="BU19000001"):
    indir_path = Path(indir)
    ndvi_path = next(indir_path.glob(f"*{target_name}*_NDVI.tif"), None)
    chm_path = next(indir_path.glob(f"*{target_name}*_CHM.tif"), None)
    cir_path = next(indir_path.glob(f"*{target_name}*_CIR.tif"), None)

    ## extra features
    bgt_cat_path = next(indir_path.glob(f"*{target_name}*_BGT_CAT.tif"), None)
    gevoelstemp_2022 = next(indir_path.glob(f"*{target_name}*_KEA_CAT_gevoelstemperatuur_2022.tif"), None)
    landbedekking_2024 = next(indir_path.glob(f"*{target_name}*_KEA_CAT_BGV_landbedekking_2024.tif"), None)

    print(f"Loading data for {target_name}...")

    # -----------------------------
    # NDVI
    # -----------------------------
    with rio.open(ndvi_path) as src:
        ndvi = src.read(1)
        profile = src.profile
        height, width = ndvi.shape
        ndvi_mask = (ndvi == 255)

    # -----------------------------
    # CHM
    # -----------------------------
    with rio.open(chm_path) as src:
        chm = src.read(1, out_shape=(height, width))
        chm_mask = (chm < -9000)

    # -----------------------------
    # CIR (NIR + RED)
    # -----------------------------
    with rio.open(cir_path) as src:
        nir = src.read(1, out_shape=(height, width))
        red = src.read(2, out_shape=(height, width))
        cir_nodata = src.nodata or 255
        cir_mask = (nir == cir_nodata) | (red == cir_nodata)

    # -----------------------------
    # BGT CAT
    # -----------------------------
    with rio.open(bgt_cat_path) as src:
        bgt_cat = src.read(1)
        profile = src.profile
        height, width = bgt_cat.shape
        bgt_cat_mask = (bgt_cat == src.nodata)

    # -----------------------------
    # KEA CAT gevoelstemp 2022
    # -----------------------------
    with rio.open(gevoelstemp_2022) as src:
        gevoelstemp = src.read(1)
        profile = src.profile
        height, width = gevoelstemp.shape
        gevoelstemp_mask = (gevoelstemp == src.nodata)

    # -----------------------------
    # KEA CAT BGV landbedekking 2024
    # -----------------------------
    with rio.open(landbedekking_2024) as src:
        landbedekking = src.read(1)
        profile = src.profile
        height, width = landbedekking.shape
        landbedekking_mask = (landbedekking == src.nodata)

    # -----------------------------
    # Combined mask
    # -----------------------------
    invalid_mask = ndvi_mask | chm_mask | cir_mask | bgt_cat_mask | gevoelstemp_mask | landbedekking_mask
    valid_indices = np.where(~invalid_mask)

    print(f"Total valid pixels: {len(valid_indices[0])}")

    # -----------------------------
    # Feature matrix
    # -----------------------------
    features = np.column_stack([
        ndvi[valid_indices],
        chm[valid_indices],
        nir[valid_indices],
        red[valid_indices],
        bgt_cat[valid_indices],
        gevoelstemp[valid_indices],
        landbedekking[valid_indices]
    ]).astype(np.float32)

    return features, valid_indices, invalid_mask, (height, width), profile

In [ ]:
features, valid_indices, invalid_mask, (height, width), profile = load_features()

Loading data for BU19000001...
Total valid pixels: 27892667


In [ ]:
features.shape

(27892667, 7)

### Reduce Pixel (superpixel)

In [ ]:
"""
Test methode (niet gebruikt!) om de pixels samen te voegen tot superpixels, hiermee verminderen we het totaal aantal pixels
en dus ook de trainingsdata.
"""

In [ ]:
def process_neighborhood_superpixel(indir, target_name):
    indir_path = Path(indir)
    ndvi_path = next(indir_path.glob(f"*{target_name}*_NDVI.tif"), None)
    chm_path = next(indir_path.glob(f"*{target_name}*_CHM.tif"), None)
    cir_path = next(indir_path.glob(f"*{target_name}*_CIR.tif"), None)

    print(f"Loading data for {target_name}...")

    # -----------------------------
    # NDVI
    # -----------------------------
    with rio.open(ndvi_path) as src:
        ndvi = src.read(1)
        profile = src.profile
        height, width = ndvi.shape
        ndvi_mask = (ndvi == 255)

    # -----------------------------
    # CHM
    # -----------------------------
    with rio.open(chm_path) as src:
        chm = src.read(1, out_shape=(height, width))
        chm_mask = (chm < -9000)

    # -----------------------------
    # CIR (NIR + RED)
    # -----------------------------
    with rio.open(cir_path) as src:
        nir = src.read(1, out_shape=(height, width))
        red = src.read(2, out_shape=(height, width))
        cir_nodata = src.nodata or 255
        cir_mask = (nir == cir_nodata) | (red == cir_nodata)

    # -----------------------------
    # Combined mask
    # -----------------------------
    invalid_mask = ndvi_mask | chm_mask | cir_mask
    valid_indices = np.where(~invalid_mask)

    print(f"Total valid pixels: {len(valid_indices[0])}")

    # -----------------------------
    # Feature matrix
    # -----------------------------
    features = np.column_stack([
        ndvi[valid_indices],
        chm[valid_indices],
        nir[valid_indices],
        red[valid_indices]
    ]).astype(np.float32)

    # Maak full raster stack (met NaNs voor invalid)
    X_full = np.stack([ndvi, chm, nir, red], axis=-1).astype(np.float32)
    X_full[invalid_mask] = np.nan

    return features, valid_indices, invalid_mask, (height, width), profile, X_full

In [ ]:
def generate_superpixels(X_full, invalid_mask, n_segments=200000, compactness=10):
    print(f"{'='*40}")
    print("Generating Superpixels (SLIC)")
    print(f"{'='*40}")
    print(f"n_segments={n_segments}, compactness={compactness}")

    # SLIC kan niet met NaNs → tijdelijk vullen
    X_temp = np.copy(X_full)
    nan_mask = np.isnan(X_temp)
    X_temp[nan_mask] = 0

    segments = slic(
        X_temp,
        n_segments=n_segments,
        compactness=compactness,
        channel_axis=-1,
        start_label=0
    )

    # Zet invalid pixels op -1 segment
    segments[invalid_mask] = -1

    print(f"Generated {segments.max()+1} superpixels")

    return segments


def aggregate_superpixel_features(X_full, segments):
    print(f"{'='*40}")
    print("Aggregating Superpixel Features")
    print(f"{'='*40}")

    valid_seg_mask = segments >= 0
    seg_ids = np.unique(segments[valid_seg_mask])

    n_segments = len(seg_ids)
    print(f"Valid superpixels: {n_segments}")

    features = np.vstack([
        mean(X_full[..., i], labels=segments, index=seg_ids)
        for i in range(X_full.shape[-1])
    ]).T

    return features.astype(np.float32), seg_ids

In [ ]:
features, valid_indices, invalid_mask, shape, profile, x_full = process_neighborhood_superpixel('/content/Groenbeeld/outdir/municipality_survey_v4', 'BU19000001')

Loading data for BU19000001...
Total valid pixels: 29657847


In [ ]:
segments = generate_superpixels(
    x_full,
    invalid_mask,
    n_segments=500000,
    compactness=4
)

Generating Superpixels (SLIC)
n_segments=500000, compactness=4
Generated 462393 superpixels


In [ ]:
sp_features, seg_ids = aggregate_superpixel_features(x_full, segments)

Aggregating Superpixel Features
Valid superpixels: 381213


In [ ]:
sp_features.shape

(381213, 4)